# Stop In-Progress Fabric Workloads Across Tenant

This notebook uses the Fabric Activity Events API to discover all running notebook sessions, notebook jobs, and pipeline jobs across the entire tenant.

It excludes configured admin workspace names from stop/cancel operations, then continuously monitors and stops new activity starting within the 30-minute window.

In [ ]:
# =========================
# CONFIGURATION
# =========================

# Optional: set for logging/traceability only. Authentication comes from notebook identity.
TARGET_TENANT_ID = "<tenant-guid-or-name>"

# Workspace display names to skip entirely (case-insensitive).
EXCLUDED_WORKSPACE_NAMES = {
    "Admin",
    "Fabric Admin"
}

# Status values to stop (normalized to lowercase).
TARGET_STATUSES = {"inprogress", "starting", "unknown", "running"}

# Safety switch. Set to False to execute actual stop/cancel calls.
DRY_RUN = True

# Continuous monitoring loop configuration
LOOP_DURATION_MINUTES = 30
POLL_INTERVAL_SECONDS = 30

# API roots
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
PBI_ADMIN_BASE = "https://api.powerbi.com/v1.0/myorg/admin"

In [ ]:
import requests
import time
from typing import Dict, List, Iterable
from notebookutils import mssparkutils

fabric_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")

fabric_headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json"
}

pbi_headers = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type": "application/json"
}

def normalize_status(value: str) -> str:
    if value is None:
        return "unknown"
    return str(value).replace(" ", "").replace("_", "").lower()

def request_json(method: str, url: str, headers: Dict[str, str], body: Dict = None, retries: int = 5):
    """
    Make an HTTP request with exponential backoff for 429 throttling.
    """
    for attempt in range(1, retries + 1):
        response = requests.request(method=method, url=url, headers=headers, json=body, timeout=60)
        
        if response.status_code == 429:
            # Use Retry-After header if available, otherwise exponential backoff
            retry_after = response.headers.get("Retry-After")
            if retry_after:
                try:
                    wait_seconds = int(retry_after)
                except ValueError:
                    wait_seconds = min(2 ** attempt, 60)  # Exponential backoff capped at 60s
            else:
                wait_seconds = min(2 ** attempt, 60)  # Exponential backoff: 2, 4, 8, 16, 32, 60...
            
            print(f"  [429 Throttled] Waiting {wait_seconds}s before retry (attempt {attempt}/{retries})...")
            time.sleep(wait_seconds)
            continue

        if response.status_code >= 400:
            return None, response

        if response.text and response.text.strip():
            return response.json(), response

        return {}, response

    return None, response

def get_all_pages(start_url: str, headers: Dict[str, str]) -> List[Dict]:
    items: List[Dict] = []
    next_url = start_url

    while next_url:
        data, resp = request_json("GET", next_url, headers)
        if resp is None or resp.status_code >= 400 or data is None:
            code = resp.status_code if resp is not None else "N/A"
            message = resp.text[:400] if resp is not None else "No response"
            raise RuntimeError(f"Failed GET {next_url}. status={code}. body={message}")

        items.extend(data.get("value", []))
        next_url = data.get("continuationUri")

    return items

def safe_post(url: str, headers: Dict[str, str], body: Dict = None) -> bool:
    if DRY_RUN:
        print(f"  [DRY_RUN] POST {url}")
        return True

    _, resp = request_json("POST", url, headers, body=body)
    if resp is None:
        print(f"  Failed POST {url}: no response")
        return False

    if resp.status_code in (200, 202, 204):
        return True

    print(f"  Failed POST {url}: {resp.status_code} {resp.text[:300]}")
    return False

In [ ]:
# =========================
# BUILD WORKSPACE NAME-TO-ID MAPPING AND EXCLUDED SET
# =========================

print("Building workspace mapping...")

# Get all groups to map workspace names to IDs
admin_groups_url = f"{PBI_ADMIN_BASE}/groups?$top=5000"
all_groups = get_all_pages(admin_groups_url, pbi_headers)

excluded_lower = {x.strip().lower() for x in EXCLUDED_WORKSPACE_NAMES}
workspace_name_to_id: Dict[str, str] = {}
excluded_workspace_ids: set = set()

for g in all_groups:
    name = g.get("name") or g.get("displayName") or ""
    ws_id = g.get("id")
    
    if name and ws_id:
        workspace_name_to_id[name.lower()] = ws_id
        
        if name.strip().lower() in excluded_lower:
            excluded_workspace_ids.add(ws_id)
            print(f"  Marked excluded workspace: {name} ({ws_id})")

print(f"Workspaces in tenant: {len(all_groups)}")
print(f"Excluded workspace IDs: {len(excluded_workspace_ids)}")
print(f"Tenant hint: {TARGET_TENANT_ID}")

In [ ]:
# =========================
# QUERY ACTIVITY EVENTS AND BUILD RUNNING ITEMS LIST
# =========================

from datetime import datetime, timedelta
import urllib.parse

def get_all_activities() -> List[Dict]:
    """
    Query Activity Events API for all activities.
    Uses date filters to get recent activities (last 24 hours).
    """
    # Activity Events API requires startDateTime and endDateTime parameters
    end_time = datetime.utcnow()
    start_time = end_time - timedelta(hours=24)  # Last 24 hours of activity
    
    # Format as ISO 8601 for API
    start_dt_str = start_time.isoformat() + "Z"
    end_dt_str = end_time.isoformat() + "Z"
    
    # Build query with filters
    activity_url = f"{PBI_ADMIN_BASE}/activityevents?startDateTime='{start_dt_str}'&endDateTime='{end_dt_str}'&$top=5000"
    
    try:
        print(f"Fetching activities from {start_dt_str} to {end_dt_str}...")
        activities = get_all_pages(activity_url, pbi_headers)
        return activities
    except Exception as ex:
        print(f"Failed to fetch activity events: {ex}")
        
        # Attempt alternative endpoint without strict date filtering
        print("Attempting alternative activity query...")
        alt_url = f"{PBI_ADMIN_BASE}/activityevents?$top=5000"
        try:
            activities = get_all_pages(alt_url, pbi_headers)
            return activities
        except Exception as ex2:
            print(f"Alternative query also failed: {ex2}")
            return []

def filter_running_activities(activities: List[Dict]) -> List[Dict]:
    """
    Filter activities to keep only running notebook sessions, notebook jobs, and pipeline jobs.
    Exclude activities from admin workspaces.
    """
    running_items = []
    
    for activity in activities:
        # Get workspace and item info
        workspace_id = activity.get("WorkspaceId")
        
        # Skip if workspace is in excluded list
        if workspace_id in excluded_workspace_ids:
            continue
        
        activity_type = activity.get("Activity") or ""
        status = normalize_status(activity.get("Status") or "")
        
        # Only process target activity types and statuses
        if status not in TARGET_STATUSES:
            continue
        
        # Map activity types to our target types
        if activity_type in ("UpdateNotebook", "ExecuteNotebookJob", "StartNotebookSession", "RunNotebook"):
            running_items.append({
                "type": "notebook",
                "workspaceId": workspace_id,
                "itemId": activity.get("ItemId"),
                "itemName": activity.get("ItemName") or activity.get("ObjectId"),
                "activityId": activity.get("Id"),
                "status": status,
                "activity": activity_type,
                "timestamp": activity.get("EventTime")
            })
        elif activity_type in ("ExecutePipeline", "RunDataPipeline", "StartPipeline", "DataPipelineRun"):
            running_items.append({
                "type": "pipeline",
                "workspaceId": workspace_id,
                "itemId": activity.get("ItemId"),
                "itemName": activity.get("ItemName") or activity.get("ObjectId"),
                "activityId": activity.get("Id"),
                "status": status,
                "activity": activity_type,
                "timestamp": activity.get("EventTime")
            })
    
    return running_items

In [ ]:
# =========================
# STOP/CANCEL ITEMS FROM ACTIVITY EVENTS (CONTINUOUS LOOP)
# =========================

def stop_notebook_session_from_activity(workspace_id: str, activity: Dict) -> bool:
    """
    Stop a notebook Spark session using workspace/notebook IDs from activity.
    """
    notebook_id = activity.get("itemId")
    session_id = activity.get("activityId")
    
    if not notebook_id or not session_id:
        return False
    
    stop_url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/notebooks/{notebook_id}/sessions/{session_id}/stop"
    return safe_post(stop_url, fabric_headers)

def cancel_item_job_from_activity(workspace_id: str, activity: Dict) -> bool:
    """
    Cancel a notebook or pipeline job using IDs from activity.
    """
    item_id = activity.get("itemId")
    job_id = activity.get("activityId")
    
    if not item_id or not job_id:
        return False
    
    # Try multiple endpoint patterns
    urls = [
        f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{item_id}/jobs/instances/{job_id}/cancel",
        f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{item_id}/jobInstances/{job_id}/cancel"
    ]
    
    for url in urls:
        if safe_post(url, fabric_headers):
            return True
    
    return False

summary = {
    "iterations": 0,
    "activities_queried_total": 0,
    "running_items_found_total": 0,
    "notebook_sessions_stopped_total": 0,
    "jobs_cancelled_total": 0,
    "errors": 0
}

LOOP_DURATION_SECONDS = LOOP_DURATION_MINUTES * 60
loop_started_at = time.time()
loop_ends_at = loop_started_at + LOOP_DURATION_SECONDS
iteration = 0

print(f"\nStarting continuous Activity Events monitoring loop for {LOOP_DURATION_MINUTES} minute(s).")
print(f"Poll interval: {POLL_INTERVAL_SECONDS}s\n")

while time.time() < loop_ends_at:
    iteration += 1
    now = time.time()
    remaining_seconds = max(0, int(loop_ends_at - now))
    
    iteration_summary = {
        "activities_queried": 0,
        "running_items_found": 0,
        "sessions_stopped": 0,
        "jobs_cancelled": 0,
        "errors": 0
    }

    print("=" * 100)
    print(f"Loop iteration {iteration} | Time remaining: {remaining_seconds}s")
    
    try:
        # Query all activities across tenant
        all_activities = get_all_activities()
        iteration_summary["activities_queried"] = len(all_activities)
        summary["activities_queried_total"] += len(all_activities)
        
        # Filter to running items, excluding admin workspaces
        running_items = filter_running_activities(all_activities)
        iteration_summary["running_items_found"] = len(running_items)
        summary["running_items_found_total"] += len(running_items)
        
        if len(running_items) == 0:
            print("No running items found (or all excluded).")
        else:
            print(f"Found {len(running_items)} running item(s) to process:\n")
            
            for item in running_items:
                item_type = item.get("type")
                workspace_id = item.get("workspaceId")
                item_name = item.get("itemName", "unknown")
                status = item.get("status")
                activity_type = item.get("activity")
                
                print(f"  Item: {item_name} | Type: {item_type} | Status: {status} | Activity: {activity_type}")
                
                # Attempt to stop/cancel
                success = False
                if item_type == "notebook":
                    success = stop_notebook_session_from_activity(workspace_id, item)
                    if success:
                        iteration_summary["sessions_stopped"] += 1
                        summary["notebook_sessions_stopped_total"] += 1
                elif item_type == "pipeline":
                    success = cancel_item_job_from_activity(workspace_id, item)
                    if success:
                        iteration_summary["jobs_cancelled"] += 1
                        summary["jobs_cancelled_total"] += 1
                
                action_result = "✓ Stopped" if success else "✗ Failed"
                print(f"    {action_result}\n")
    
    except Exception as ex:
        print(f"ERROR during iteration: {ex}")
        iteration_summary["errors"] += 1
        summary["errors"] += 1
    
    summary["iterations"] += 1
    
    print(f"\nIteration {iteration} Summary:")
    for k, v in iteration_summary.items():
        print(f"  {k}: {v}")
    
    if time.time() >= loop_ends_at:
        break
    
    sleep_seconds = min(POLL_INTERVAL_SECONDS, max(0, int(loop_ends_at - time.time())))
    if sleep_seconds > 0:
        print(f"\nSleeping {sleep_seconds}s before next scan...")
        time.sleep(sleep_seconds)

print("\n" + "=" * 100)
print("Activity Events monitoring loop completed.")

In [ ]:
total_elapsed_seconds = int(time.time() - loop_started_at)

print("#" * 100)
print("FINAL EXECUTION SUMMARY")
print("#" * 100)
print(f"Monitoring window: {LOOP_DURATION_MINUTES} minute(s)")
print(f"Actual elapsed time: {total_elapsed_seconds}s")
print(f"Loop iterations: {summary['iterations']}")
print(f"\nActivity Events:")
print(f"  Total activities queried across all iterations: {summary['activities_queried_total']}")
print(f"  Total running items identified (excluding admin): {summary['running_items_found_total']}")
print(f"\nActions Taken:")
print(f"  Notebook sessions stopped: {summary['notebook_sessions_stopped_total']}")
print(f"  Jobs cancelled (pipelines): {summary['jobs_cancelled_total']}")
print(f"  Errors encountered: {summary['errors']}")
print(f"\nExcluded workspaces: {sorted(EXCLUDED_WORKSPACE_NAMES)}")
print(f"\nMode: {'DRY_RUN (no changes)' if DRY_RUN else 'LIVE (changes applied)'}")